# Self-Guided Lab: Retrieval-Augmented Generation (RAG) — Completed

All **four required exercises** are completed. **Bonus excluded.**

**Before running:** Put the course PDF at `LAB/ai-for-everyone.pdf` relative to this notebook (or change `file_path` below). Set `OPENAI_API_KEY` in your environment or in a local `.env` file. The PDF and API key were not included with the lab prompt, so the live answers cannot be precomputed.

> This lab uses OpenAI embeddings and a GPT model, which may incur API costs. Run notebook cells from top to bottom.


## 1. Install and import packages

In [1]:
%pip install -q -U langchain-community langchain-text-splitters langchain-openai langchain-chroma pypdf python-dotenv termcolor chromadb

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from termcolor import colored
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from openai import OpenAI

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY is missing. Set it in your environment or .env file.")


## 2. Load PDF pages and split into chunks

In [ ]:
file_path = Path("LAB/ai-for-everyone.pdf")
if not file_path.is_file():
    raise FileNotFoundError(f"PDF not found: {file_path.resolve()}. Update file_path to your book PDF.")

loader = PyPDFLoader(str(file_path))
pages = loader.load()  # Each page becomes a LangChain Document
print("Number of pages:", len(pages))

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)
print("Number of chunks:", len(chunks))
print("First chunk preview:\n", chunks[0].page_content[:500] if chunks else "No text extracted")


**Note:** `CharacterTextSplitter` is an alternative, but we use `RecursiveCharacterTextSplitter` to keep related content together when possible.

## 3. Create embeddings and Chroma vector database

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# Index this PDF's text in a local Chroma collection.
# A new local directory is used so existing stored collections remain untouched.
db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="ai_for_everyone_lab",
    persist_directory="./chroma_db_LAB",
)
print("ChromaDB created with document embeddings.")


## Exercise 1 — Ask a question about the book

In [ ]:
# This book is about AI for a general audience.
user_question = "What is artificial intelligence, and how can it be used in everyday life?"
retrieved_docs = db.similarity_search(user_question, k=10)

print("Question:", user_question)
print("Retrieved documents:", len(retrieved_docs))
for i, doc in enumerate(retrieved_docs[:3], start=1):
    page = doc.metadata.get("page")
    page_label = f"page {page + 1}" if isinstance(page, int) else "page unknown"
    print(f"\nDocument {i} ({page_label}):\n{doc.page_content[:1000]}")


## 4. Prepare retrieved context

In [ ]:
def _get_document_prompt(docs):
    return "\n\n".join(
        f"Source {i + 1} (page {doc.metadata.get('page', 'unknown')}):\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )

formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")


## Exercise 2 — Write a book-appropriate prompt

In [ ]:
prompt = f"""You are a friendly educational assistant helping someone understand the book
'AI for Everyone'. Explain ideas in simple, clear language suitable for beginners.

Answer the question using only the book excerpts provided below. Give an easy-to-follow
explanation and, if supported by the context, one practical example. If the excerpts do
not contain enough information, say that the available text does not answer the question.
Do not invent facts or quotes from the book.

Book excerpts:
{formatted_context}

Question: {user_question}

Answer:"""
print(prompt[:1000] + ("..." if len(prompt) > 1000 else ""))


## Exercise 3 — Configure GPT response parameters

In [ ]:
client = OpenAI()
model_params = {
    "model": "gpt-4o",
    "temperature": 0.3,         # Low: more focused, grounded responses
    "max_tokens": 500,          # Enough for a short explanation
    "top_p": 1.0,               # Do not further restrict sampling
    "frequency_penalty": 0.2,   # Slightly discourage repetition
    "presence_penalty": 0.0,    # Do not encourage unrelated topics
}

completion = client.chat.completions.create(
    messages=[{"role": "user", "content": prompt}],
    **model_params,
    timeout=120,
)
answer = completion.choices[0].message.content
print(answer)


## 5. Keyword highlighting — Exercise 4

In [ ]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, "green", attrs=["bold"]))
    return text

# Keywords tied to the chosen user question
query_keywords = ["artificial intelligence", "AI", "everyday", "technology", "machine learning"]

for i, doc in enumerate(retrieved_docs[:1], start=1):
    snippet = doc.page_content[:200]
    highlighted = highlight_keywords(snippet, query_keywords)
    print(f"Snippet {i}:\n{highlighted}\n{'-' * 80}")


## Conclusion

The completed RAG workflow loads a PDF, splits it into chunks, generates OpenAI embeddings, stores them in Chroma, retrieves relevant chunks, formats retrieved text as context, answers with GPT, and highlights query-related keywords.

**Exercises completed:** 1 (question), 2 (prompt), 3 (parameters), 4 (keywords). **Bonus not included.**

*Results are generated when you run the notebook with the actual PDF and a working API key.*